# PM-RL1 数据与测量诊断

全量结构统计覆盖三个公开来源及当前派生数据；train/dev 内容核查、test 仅结构统计。词汇标记不是错误标签。此 notebook 只读取本次统计，不将 gold 送入训练或运行时。


In [1]:
from pathlib import Path
import json, csv
relative = Path('project/docs/reviews/20260930/pm_rl1/dataset_diagnostics_20260930_v1')
OUT = next((p if (p/'census.json').is_file() else p/relative) for p in (Path.cwd(), *Path.cwd().parents) if (p/'census.json').is_file() or (p/relative/'census.json').is_file())
def read(name): return json.loads((OUT/name).read_text())
s = read('census.json')
c = read('capacity.json')
print('Loaded published research summaries')


Loaded published research summaries


## 来源与统计单位

18 个构造人物不等于独立真实参与者；同源会话和同一前缀不同资源条件不应当作独立样本。中文 ESC 卡片只是库存，不属于当前英文主评测。


In [2]:
print('ESConv:', s['esconv']['dialogues'], 'dialogues;', s['esconv']['turns'], 'turns')
print('EvoEmo:', s['history']['owners'], 'owners;', s['history']['sessions'], 'sessions')
print('ES-MemEval by split:', s['tasks'])
print('ESC English:', s['esc_eval']['en']['cards'], s['esc_eval']['en']['sources'])
print('MP/ME:', s['memory']['head_counts'])


ESConv: 1300 dialogues; 38365 turns
EvoEmo: 18 owners; 401 sessions
ES-MemEval by split: {'train': {'qa': 1003, 'summary': 81, 'dg': 25}, 'dev': {'qa': 258, 'summary': 22, 'dg': 4}, 'test': {'qa': 166, 'summary': 22, 'dg': 5}}
ESC English: 331 {'ESconv': 158, 'EPITOME': 5, 'ExTES': 70, 'MHP': 73, 'Psych': 25}
MP/ME: {'MP': 713, 'ME': 1523}


## 角色与记忆状态

Self-disclosure 是原人类数据的策略类别；AI 不能据此继承真人生活经历。引文字符准确只说明来源可追溯，不说明状态仍有效。


In [3]:
print('ESConv self-disclosure turns:', s['esconv']['strategy_counts']['Self-disclosure'])
print('RS self-disclosure:', s['rs']['self_disclosure_treatments'], '/', s['rs']['treatments'])
print('Top-four exposure:', s['rs']['top4_prefixes_with_self_disclosure'], '/', s['prefixes']['rows'])
print('Memory:', s['memory'])


ESConv self-disclosure turns: 1713
RS self-disclosure: 1337 / 13172
Top-four exposure: 45 / 114
Memory: {'units': 2236, 'head_counts': {'MP': 713, 'ME': 1523}, 'types': {'durable_health_constraint': 176, 'enduring_preference': 93, 'stable_relationship_role': 190, 'coping_attempt': 374, 'relationship_event': 293, 'transient_state': 84, 'meaningful_experience': 314, 'occupation': 40, 'enduring_interest': 134, 'state_onset': 111, 'life_event': 148, 'support_experience': 136, 'state_change': 57, 'durable_boundary': 19, 'identity': 8, 'action_observed_outcome': 4, 'education': 43, 'other_occurred_event': 2, 'location': 10}, 'temporal_status': {'ongoing_stage': 475, 'occurred': 1048}, 'missing_turn_refs': 0, 'bad_span_slices': 0, 'only_supporter_units': 0, 'repeat_same_slot_value': 201, 'reviewed_repeat_units': 101, 'overlay_reviewed': 101, 'overlay_status': {'citation_unrelated': 26, 'citation_partial': 60, 'citation_sufficient': 13, 'citation_conflicts_with_current_slot': 2}, 'temporal_blo

## QA 与可见证据

以下是引用结构分层，不是可信标签数量。只有私有事件引用也可能在可见对话中另有证据；Unknown 需要范围判断。正式公开评测集仍保持原样。


In [4]:
a=read('audit_readiness.json')
assert sum(a['qa_structural_strata'].values()) == 1261
print(a)
print('Semantic examples:', len(read('semantic_review_findings.json')['rows']))


{'qa_structural_strata': {'visible_reference_present': 997, 'no_evidence': 224, 'private_only': 20, 'unresolved_reference': 20}, 'semantic_sample_rows': 20, 'semantic_sample_selection': 'two per split x capability; illustrative audit, not quality prevalence', 'unresolved_occurrences': 30, 'occurrences_with_owner_local_candidate': 16, 'automatic_repairs_applied': 0, 'me_outcome_text_nonempty': 7, 'current_q_scope': 'Official full owner history; question group is not a chronological cutoff and is not exposed as a hidden-answer hint.', 'proposed_auxiliary_use': 'None admitted by syntax checks alone. Any new as-of QA auxiliary set requires separately versioned visible evidence, time anchor and answer-scope review.'}
Semantic examples: 20


## 可达动作与训练覆盖

所有 114 个前缀都允许某些 3/4 次获取或同类多取；训练覆盖缺口不能通过给裁判调阈值补齐。


In [5]:
print('Capacity:', c)
print('SFT:', s['sft'])
assert sum(c['by_acquisition_count'].values()) == c['total_reachable']
assert s['sft']['train']['rows'] + s['sft']['dev']['rows'] == 155


Capacity: {'prefixes': 114, 'reachable_plans': {'n': 114, 'min': 16, 'median': 68.0, 'p90': 69, 'max': 70}, 'total_reachable': 7240, 'by_acquisition_count': {'0': 114, '1': 447, '2': 1094, '3': 2116, '4': 3469}, 'prefixes_with_three': 114, 'prefixes_with_four': 114, 'prefixes_with_same_head_repeat': 114, 'first_head_unreachable': {'RS': 0, 'MP': 8, 'MS': 0, 'ME': 1}, 'full_history_text_tokens': {'n': 18, 'min': 8066, 'median': 13472.5, 'p90': 20489, 'max': 20713}, 'note': 'Reachability recomputed by actual ResourceEnv and full-resource renderer; no G/J calls. Text-only history tokens exclude judge template/replies.', 'tokenizer_json_sha256': '79e3e522635f3171300913bb421464a87de6222182a0570b9b2ccba2a964b2b4'}
SFT: {'train': {'rows': 103, 'owners': 12, 'prefixes': 19, 'resource_counts': {'0': 34, '1': 59, '2': 10}, 'same_head_repeat_rows': 0, 'response_words': {'n': 103, 'min': 7, 'median': 67, 'p90': 105, 'max': 179}, 'response_self_life_candidates': 0, 'prefix_self_life_candidates': 0,

## 复算

在新的输出目录依次执行 69、70、71 脚本的 `--out` 参数。70 只加载本地 tokenizer，不加载模型权重。原数据、split、gold 和旧输出均不改。20 个示例的判断由 coordinator 核查，不是新的人评或训练标签。

```bash
cd /home/tokkio/Metacom5_3
/opt/tokkio-data0/tokkio_envs/paper1-py311/bin/python project/scripts/rl1/69_analyze_dataset_features.py --out /tmp/pmrl1_dataset_recompute
/opt/tokkio-data0/tokkio_envs/paper1-py311/bin/python project/scripts/rl1/70_analyze_dataset_capacity.py --out /tmp/pmrl1_dataset_recompute
/opt/tokkio-data0/tokkio_envs/paper1-py311/bin/python project/scripts/rl1/71_prepare_dataset_evidence_audit.py --out /tmp/pmrl1_dataset_recompute
```
